# Tarefa 2 - Baseline e rotulagem

Este notebook executa a lógica de baseline por fluxo, cálculo das métricas do Período B, rotulagem em OK/RISCO/FALHA e recorte em treino/validação/teste.

Estrutura sugerida: coleta v2 -> raw -> notebook de rotulagem -> CSVs finais.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
ROOT = next(
    (
        candidate
        for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (candidate / 'data_ripe_atlas' / 'raw' / 'v2').is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Não foi possível localizar data_ripe_atlas/raw/v2 a partir do diretório atual.')

RAW = ROOT / 'data_ripe_atlas' / 'raw' / 'v2'
PROCESSED = ROOT / 'data_ripe_atlas' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

CSV_PERIODO_A = RAW / 'ripe_atlas_mesh_v2_ping_ipv4_periodo_a.csv'
CSV_PERIODO_B = RAW / 'ripe_atlas_mesh_v2_ping_ipv4_periodo_b.csv'
print('Arquivos esperados:', CSV_PERIODO_A.exists(), CSV_PERIODO_B.exists())


Arquivos esperados: True True


## 1. Carregar dados

Os CSVs da coleta v2 já devem existir na pasta `data_ripe_atlas/raw/v2/` antes de rodar esta etapa.

In [12]:
# 1. Leitura dos dados de entrada
# O Período A é usado para congelar o baseline do fluxo.
# O Período B é usado para calcular as métricas relativas e rotular a classe.

df_a = pd.read_csv(CSV_PERIODO_A)
df_b = pd.read_csv(CSV_PERIODO_B)

# Visualização inicial do schema para conferir se a leitura do CSV está correta.
df_a.head()


,fluxo_id,probe_id,dst_addr,measurement_id,timestamp,datetime_utc,latencia_ms,rtt_min_ms,rtt_max_ms,enviados,recebidos,perda_pct,jitter_ms,timeout_atual,pais_origem,pais_destino,rota,periodo
0,6659|148.163.220.5,6659,148.163.220.5,22680685,1788739270,2026-09-07T00:01:10+00:00,0.945870,0.599970,1.252377,3,3,0.0,0.3280,0,BR,BR,BR→BR,A
1,6659|148.163.220.5,6659,148.163.220.5,22680685,1788739509,2026-09-07T00:05:09+00:00,1.355784,0.718796,1.889908,3,3,0.0,0.5923,0,BR,BR,BR→BR,A
2,6659|148.163.220.5,6659,148.163.220.5,22680685,1788739746,2026-09-07T00:09:06+00:00,0.917313,0.876387,0.974230,3,3,0.0,0.0508,0,BR,BR,BR→BR,A
3,6659|148.163.220.5,6659,148.163.220.5,22680685,1788739986,2026-09-07T00:13:06+00:00,0.785229,0.663937,0.918060,3,3,0.0,0.1275,0,BR,BR,BR→BR,A
4,6659|148.163.220.5,6659,148.163.220.5,22680685,1788740231,2026-09-07T00:17:11+00:00,0.975084,0.875632,1.064885,3,3,0.0,0.0950,0,BR,BR,BR→BR,A


In [13]:

df_b.head()


,fluxo_id,probe_id,dst_addr,measurement_id,timestamp,datetime_utc,latencia_ms,rtt_min_ms,rtt_max_ms,enviados,recebidos,perda_pct,jitter_ms,timeout_atual,pais_origem,pais_destino,rota,periodo
0,6659|148.163.220.5,6659,148.163.220.5,22680685,1789344070,2026-09-14T00:01:10+00:00,0.776950,0.580753,1.104954,3,3,0.0,0.2859,0,BR,BR,BR→BR,B
1,6659|148.163.220.5,6659,148.163.220.5,22680685,1789344303,2026-09-14T00:05:03+00:00,0.845547,0.668828,1.034297,3,3,0.0,0.1830,0,BR,BR,BR→BR,B
2,6659|148.163.220.5,6659,148.163.220.5,22680685,1789344541,2026-09-14T00:09:01+00:00,0.725460,0.613564,0.821164,3,3,0.0,0.1047,0,BR,BR,BR→BR,B
3,6659|148.163.220.5,6659,148.163.220.5,22680685,1789344784,2026-09-14T00:13:04+00:00,0.552379,0.466098,0.635651,3,3,0.0,0.0848,0,BR,BR,BR→BR,B
4,6659|148.163.220.5,6659,148.163.220.5,22680685,1789345031,2026-09-14T00:17:11+00:00,0.974101,0.733649,1.438735,3,3,0.0,0.4025,0,BR,BR,BR→BR,B


## 2. Normalizar colunas para o padrão do projeto

A coleta v2 usa nomes diferentes dos do dicionário v0.1; aqui fazemos o mapeamento para manter a lógica do script consistente.

In [14]:
# 2. Padronização das colunas para o padrão do projeto
# Esta etapa faz o mapeamento do v2 para as colunas do dicionário do projeto.
# Isso evita que o restante das operações dependam do nome da coluna original da coleta.
COLUNAS_MAP = {
    'latencia_ms': 'avg',
    'enviados': 'sent',
    'recebidos': 'rcvd',
    'probe_id': 'prb_id',
    'datetime_utc': 'datetime_utc',
}

df_a = df_a.rename(columns=COLUNAS_MAP)
df_b = df_b.rename(columns=COLUNAS_MAP)

# Confirmar que os nomes finais ficaram no padrão esperado.
df_a.columns.tolist()


['fluxo_id',
 'prb_id',
 'dst_addr',
 'measurement_id',
 'timestamp',
 'datetime_utc',
 'avg',
 'rtt_min_ms',
 'rtt_max_ms',
 'sent',
 'rcvd',
 'perda_pct',
 'jitter_ms',
 'timeout_atual',
 'pais_origem',
 'pais_destino',
 'rota',
 'periodo']

## 3. Baseline do Período A

Uma ficha por fluxo (`prb_id|dst_addr`). Só vale se a quantidade de RTTs válidos no Período A for maior ou igual a 1500.

In [15]:
# 3. Cálculo do baseline do Período A
# A lógica é: para cada fluxo, calcular mediana do RTT, MAD, jitter típico e perda típica.
# Fluxos com menos de 1.500 RTTs válidos são excluídos; eles não recebem baseline nem classe.

def calcular_mad(serie):
    """Calcula o desvio absoluto mediano (MAD) do trecho do fluxo."""
    mediana = serie.median()
    return (serie - mediana).abs().median()

resultados = []
fluxos_excluidos = []

for (prb_id, dst_addr), grupo in df_a.groupby(['prb_id', 'dst_addr']):
    # Só consideramos RTT válidos, isto é, valores não nulos.
    rtt_validos = grupo['avg'].dropna()

    # Regra do piso mínimo: menos de 1.500 RTTs válidos => baseline insuficiente.
    if len(rtt_validos) < 1500:
        fluxos_excluidos.append({
            'prb_id': prb_id,
            'dst_addr': dst_addr,
            'n_validos': len(rtt_validos),
        })
        continue

    mediana = rtt_validos.median()
    mad = calcular_mad(rtt_validos)
    jitter_tipico = grupo['jitter_ms'].dropna().median() if 'jitter_ms' in grupo.columns else 0
    perda_tipica = grupo['perda_pct'].median() if 'perda_pct' in grupo.columns else 0
    prop_resposta = len(rtt_validos) / len(grupo)

    resultados.append({
        'prb_id': prb_id,
        'dst_addr': dst_addr,
        'fluxo_id': f'{prb_id}|{dst_addr}',
        'mediana': mediana,
        'MAD': mad,
        'jitter_tipico': jitter_tipico,
        'perda_tipica': perda_tipica,
        'prop_resposta': prop_resposta,
        'status': 'ok',
    })

baseline = pd.DataFrame(resultados)
baseline_excluidos = pd.DataFrame(fluxos_excluidos)

print('Fluxos com baseline válido:', len(baseline))
print('Fluxos excluídos:', len(baseline_excluidos))
baseline.head()


Fluxos com baseline válido: 76
Fluxos excluídos: 16


,prb_id,dst_addr,fluxo_id,mediana,MAD,jitter_tipico,perda_tipica,prop_resposta,status
0,6396,185.32.189.249,6396|185.32.189.249,42.259319,0.103116,0.09025,0.0,0.999206,ok
1,6396,212.119.5.254,6396|212.119.5.254,10.988284,0.061758,0.05690,0.0,1.000000,ok
2,6396,91.209.16.127,6396|91.209.16.127,48.082211,0.119324,0.13650,0.0,1.000000,ok
3,6659,102.222.103.100,6659|102.222.103.100,341.228316,0.143754,0.06090,0.0,1.000000,ok
4,6659,103.167.46.8,6659|103.167.46.8,283.268084,0.287212,0.35120,0.0,1.000000,ok


In [17]:
# Salvar o baseline congelado e a lista de fluxos excluídos para auditoria.
baseline.to_csv(PROCESSED / 'baseline_por_fluxo.csv', index=False)
baseline_excluidos.to_csv(PROCESSED / 'fluxos_excluidos_baseline.csv', index=False)
print('Baseline salvo em', 'data_ripe_atlas/processed/baseline_por_fluxo.csv')


Baseline salvo em data_ripe_atlas/processed/baseline_por_fluxo.csv


## 4. Métricas do Período B

Agora usamos o baseline do fluxo para calcular z_robusto, aumento percentual, jitter relativo e informações de janela deslizante.

In [18]:
# 4. Calcular métricas do Período B usando o baseline congelado do mesmo fluxo
# Cada linha do B é comparada ao comportamento normal do fluxo no período A.
# A ideia é medir desvio robusto versus o normal do próprio fluxo, e não comparar RTT absoluto entre rotas.

df_b = df_b.merge(
    baseline[['prb_id', 'dst_addr', 'mediana', 'MAD', 'jitter_tipico']],
    on=['prb_id', 'dst_addr'],
    how='inner'
)

# z_robusto: robusto ao desvio em relação ao baseline do fluxo.
# Se MAD = 0, usamos denom. mínimo para evitar divisão por zero.
denominador = df_b['MAD'].apply(lambda x: max(x * 1.4826, 1) if x == 0 else x * 1.4826)
df_b['z_robusto'] = (df_b['avg'] - df_b['mediana']) / denominador
df_b.loc[(df_b['MAD'] == 0) & (df_b['avg'] == df_b['mediana']), 'z_robusto'] = 0

# Aumento percentual em relação à mediana do fluxo.
df_b['aumento_pct'] = (df_b['avg'] - df_b['mediana']) / df_b['mediana'] * 100

# Jitter relativo: compara o jitter atual ao jitter típico do fluxo.
df_b['jitter_relativo'] = df_b['jitter_ms'] / df_b['jitter_tipico'].replace(0, np.nan)
df_b.loc[(df_b['jitter_tipico'] == 0) & (df_b['jitter_ms'] == 0), 'jitter_relativo'] = 1
df_b.loc[(df_b['jitter_tipico'] == 0) & (df_b['jitter_ms'] > 0), 'jitter_relativo'] = 3

# Perda percentual e timeout atual.
df_b['perda_pct'] = (df_b['sent'] - df_b['rcvd']) / df_b['sent'] * 100
df_b['timeout_atual'] = 0
df_b.loc[df_b['avg'].isna() | (df_b['perda_pct'] == 100), 'timeout_atual'] = 1

# Ordenar o fluxo em tempo para calcular janelas deslizantes nas últimas 5 medições.
df_b = df_b.sort_values(by=['prb_id', 'dst_addr', 'timestamp'])

df_b['n5_timeout'] = df_b.groupby(['prb_id', 'dst_addr'])['timeout_atual'].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum()
)

df_b['n5_aumento80'] = df_b.groupby(['prb_id', 'dst_addr'])['aumento_pct'].transform(
    lambda x: (x > 80).rolling(window=5, min_periods=1).sum()
)

# Critério individual de risco para a linha 5 da regra de rotulagem.
cond_risco_ind = (
    ((df_b['z_robusto'] >= 2) & (df_b['z_robusto'] < 3.5)) |
    ((df_b['aumento_pct'] >= 30) & (df_b['aumento_pct'] <= 80)) |
    (df_b['jitter_relativo'] >= 3)
)
df_b['crit_risco_ind'] = cond_risco_ind.astype(int)
df_b['n5_risco'] = df_b.groupby(['prb_id', 'dst_addr'])['crit_risco_ind'].transform(
    lambda x: x.rolling(window=5, min_periods=1).sum()
)

df_b[['fluxo_id', 'timestamp', 'z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco']].head()


,fluxo_id,timestamp,z_robusto,aumento_pct,jitter_relativo,perda_pct,timeout_atual,n5_timeout,n5_aumento80,n5_risco
125860,6396|185.32.189.249,1789273991,2.743731,0.992588,0.372299,0.0,0,0.0,0.0,1.0
125861,6396|185.32.189.249,1789274224,-0.165921,-0.060025,1.078116,0.0,0,0.0,0.0,1.0
125862,6396|185.32.189.249,1789274469,-0.756311,-0.273608,0.185042,0.0,0,0.0,0.0,1.0
125863,6396|185.32.189.249,1789274713,-0.334060,-0.120851,0.450970,0.0,0,0.0,0.0,1.0
125864,6396|185.32.189.249,1789274944,0.173363,0.062717,0.183934,0.0,0,0.0,0.0,1.0


## 5. Rotulagem em OK / RISCO / FALHA

A regra segue a ordem da tabela da Tarefa 2: FALHA ganha de RISCO, e RISCO ganha de OK.

In [19]:
# 5. Rotulagem em OK / RISCO / FALHA
# A regra deve ser aplicada na ordem da tabela da Tarefa 2.
# Importante: FALHA vence RISCO e RISCO vence OK, e a primeira regra verdadeira para cada linha decide a classe.

df_b_rotulado = df_b.copy()
df_b_rotulado['classe'] = 'OK'

# Linha 1: perda percentual >= 10%
df_b_rotulado.loc[df_b_rotulado['perda_pct'] >= 10, 'classe'] = 'FALHA'

# Linha 2: n5_timeout >= 3
# (somente se a linha ainda não foi rotulada como FALHA)
df_b_rotulado.loc[(df_b_rotulado['classe'] == 'OK') & (df_b_rotulado['n5_timeout'] >= 3), 'classe'] = 'FALHA'

# Linha 3: z_robusto >= 3.5 e RTT presente
df_b_rotulado.loc[(df_b_rotulado['classe'] == 'OK') & (df_b_rotulado['z_robusto'] >= 3.5) & df_b_rotulado['avg'].notna(), 'classe'] = 'FALHA'

# Linha 4: n5_aumento80 >= 2
df_b_rotulado.loc[(df_b_rotulado['classe'] == 'OK') & (df_b_rotulado['n5_aumento80'] >= 2), 'classe'] = 'FALHA'

# Linha 5: risco persistente: critério individual + n5_risco >= 2
df_b_rotulado.loc[(df_b_rotulado['classe'] == 'OK') & (df_b_rotulado['crit_risco_ind'] == 1) & (df_b_rotulado['n5_risco'] >= 2), 'classe'] = 'RISCO'

print(df_b_rotulado['classe'].value_counts())
df_b_rotulado[['fluxo_id', 'timestamp', 'classe', 'z_robusto', 'perda_pct', 'n5_timeout', 'n5_risco']].head(10)


classe
OK       142384
FALHA     32788
RISCO     15684
Name: count, dtype: int64


,fluxo_id,timestamp,classe,z_robusto,perda_pct,n5_timeout,n5_risco
125860,6396|185.32.189.249,1789273991,OK,2.743731,0.0,0.0,1.0
125861,6396|185.32.189.249,1789274224,OK,-0.165921,0.0,0.0,1.0
125862,6396|185.32.189.249,1789274469,OK,-0.756311,0.0,0.0,1.0
125863,6396|185.32.189.249,1789274713,OK,-0.334060,0.0,0.0,1.0
125864,6396|185.32.189.249,1789274944,OK,0.173363,0.0,0.0,1.0
125865,6396|185.32.189.249,1789275180,OK,-0.534141,0.0,0.0,0.0
125866,6396|185.32.189.249,1789275425,OK,-0.396462,0.0,0.0,0.0
125867,6396|185.32.189.249,1789275664,OK,0.314515,0.0,0.0,0.0
125868,6396|185.32.189.249,1789275909,FALHA,4.470763,0.0,0.0,1.0
125869,6396|185.32.189.249,1789276151,OK,-0.821931,0.0,0.0,1.0


## 6. Recorte temporal por fluxo

Treino = primeiros 50%; validação = próximos 20%; teste = restante. O corte é por fluxo e em ordem temporal.

In [20]:
# 6. Recorte temporal por fluxo
# Regra: para cada fluxo, os primeiros 50% do histórico vão para treino,
# os próximos 20% vão para validação e o restante para teste.
# Importante: a divisão é feita por fluxo, e não por linha aleatória.

partes = []
for fluxo_id, grupo in df_b_rotulado.groupby('fluxo_id', sort=False):
    n = len(grupo)
    n_treino = int(n * 0.50)
    n_validacao = int(n * 0.20)

    grupo = grupo.copy()
    grupo['conjunto'] = ['treino'] * n_treino + ['validacao'] * n_validacao + ['teste'] * (n - n_treino - n_validacao)
    partes.append(grupo)

dataset_final = pd.concat(partes, ignore_index=True)
print(dataset_final['conjunto'].value_counts())
dataset_final[['fluxo_id', 'timestamp', 'conjunto', 'classe']].head()


conjunto
treino       95406
teste        57312
validacao    38138
Name: count, dtype: int64


,fluxo_id,timestamp,conjunto,classe
0,6396|185.32.189.249,1789273991,treino,OK
1,6396|185.32.189.249,1789274224,treino,OK
2,6396|185.32.189.249,1789274469,treino,OK
3,6396|185.32.189.249,1789274713,treino,OK
4,6396|185.32.189.249,1789274944,treino,OK


In [22]:
# Exportar o dataset rotulado e os três blocos finais.
# Esses arquivos são a base de entrada para o modelo e para a validação no próximo passo.

dataset_final.to_csv(PROCESSED / 'dataset_rotulado_b.csv', index=False)
dataset_final[dataset_final['conjunto'] == 'treino'].to_csv(PROCESSED / 'treino.csv', index=False)
dataset_final[dataset_final['conjunto'] == 'validacao'].to_csv(PROCESSED / 'validacao.csv', index=False)
dataset_final[dataset_final['conjunto'] == 'teste'].to_csv(PROCESSED / 'teste.csv', index=False)

print('Arquivos exportados com sucesso.')


Arquivos exportados com sucesso.


## 7. Validação final

Checar se:
- não há linhas do Período A no treino;
- cada fluxo foi dividido em treino/validação/teste em ordem temporal;
- não há duplicatas por fluxo e timestamp;
- as classes são somente OK, RISCO e FALHA.

In [23]:
# 7. Validação final do dataset gerado
# Checagens de integridade para evitar vazamentos e inconsistências antes de seguir para treinamento.
assert not dataset_final.duplicated(subset=['fluxo_id', 'timestamp']).any()
assert set(dataset_final['classe'].unique()) <= {'OK', 'RISCO', 'FALHA'}
assert dataset_final['conjunto'].isin(['treino', 'validacao', 'teste']).all()

print('Validação final concluída com sucesso.')


Validação final concluída com sucesso.
